# LSTM Slogan Classifier and Generator

This project aims to train a Long Short-Term Memory (LSTM) model to generate slogans for businesses based on their industry, and also train a classifier to predict the industry based on a given slogan.

## Import Libraries


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy  # available on Google Colab
from sklearn.model_selection import train_test_split

## Load, Explore and Clean Dataset

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load data from Google Drive
try:
    data = pd.read_csv(
        "/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv")
except FileNotFoundError:
    print("File not found, please check the file path!")

In [ ]:
# Explore shape, dtype and head
print(f'Dataset shape: {data.shape}\n')
data.info()
print('\n')
data.head()

Dataset shape: (5346, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5346 entries, 0 to 5345
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   desc           5346 non-null   object
 1   output         5346 non-null   object
 2   type           5346 non-null   object
 3   company        5346 non-null   object
 4   industry       5346 non-null   object
 5   url            5346 non-null   object
 6   alias          4017 non-null   object
 7   desc_masked    5346 non-null   object
 8   output_masked  5346 non-null   object
 9   ent_dict       5346 non-null   object
 10  unsupported    5346 non-null   bool  
 11  first_pos      5346 non-null   object
dtypes: bool(1), object(11)
memory usage: 464.8+ KB




,desc,output,type,company,industry,url,alias,desc_masked,output_masked,ent_dict,unsupported,first_pos
0,The latest <company> & Point of Sale tech for ...,Taking Care of Small Business Technology,headline_long,eftpos warehouse,computer hardware,eftposwarehouse.co.nz,Eftpos Warehouse,The latest <company> & Point of Sale tech for ...,Taking Care of Small Business Technology,{'[date]': 'monthly'},False,VB
1,Easily deliver personalized activities that en...,Build World-Class Recreation Programs,headline,welbi,"health, wellness and fitness",welbi.co,Welbi,Easily deliver personalized activities that en...,Build World-Class Recreation Programs,{},False,VB
2,Powerful lead generation software that convert...,Most Powerful Lead Generation Software for Mar...,headline_long,optinmonster,internet,optinmonster.com,Optinmonster,Powerful lead generation software that convert...,Most Powerful Lead Generation Software for Mar...,{},False,JJ
3,Twine matches companies to the best digital an...,Hire quality freelancers for your job,headline_long,twine.fm,internet,twine.fm,NaN,Twine matches companies to the best digital an...,Hire quality freelancers for your job,"{'[number]': 'over 260,000'}",False,VB
4,"Financial Advisers Norwich, Norfolk - <company...","Financial Advisers Norwich, Norfolk",headline,mcb financial services ltd,financial services,mcbfinancialservices.co.uk,Mcb Financial Services,"Financial Advisers [country], [country1] - <co...","Financial Advisers [country], [country1]","{'[country]': 'Norwich', '[country1]': 'Norfolk'}",False,NN


In [ ]:
# Extract relevant columns to df
df = data[['industry', 'output']]

# Check df shape
print(f'Dataset shape: {df.shape}\n')

Dataset shape: (5346, 2)



In [ ]:
# Check for and remove missing values
print(f'Missing values:\n{df.isnull().sum()}')
df = df.dropna()

# Check for and remove duplicate rows
print(f'\nDuplicated rows: {df.duplicated().sum()}')
df = df.drop_duplicates()

# Check final df shape
print(f'\nDataset shape: {df.shape}\n')

Missing values:
industry    0
output      0
dtype: int64

Duplicated rows: 20

Dataset shape: (5326, 2)



The dataset was checked for missing values across the industry and output columns, and none were found. 20 duplicate rows were found and removed, reducing the dataset accordingly.

## Data Preprocessing



In [ ]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")


# Define text preprocessing function
def preprocess_text(text):
    """
    Lowercase text and strip punctuation.

    Parameters:
        text: raw slogan text to preprocess.

    Returns:
        A string of lowercased, punctuation-free tokens joined by spaces
    """
    # Lowercase text
    text_lower = text.lower()
    doc = nlp(text_lower)

    # Remove punctuation
    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)


# Preprocess df
df["processed_slogan"] = df["output"].apply(preprocess_text)
df.head()

,industry,output,processed_slogan
0,computer hardware,Taking Care of Small Business Technology,taking care of small business technology
1,"health, wellness and fitness",Build World-Class Recreation Programs,build world class recreation programs
2,internet,Most Powerful Lead Generation Software for Mar...,most powerful lead generation software for mar...
3,internet,Hire quality freelancers for your job,hire quality freelancers for your job
4,financial services,"Financial Advisers Norwich, Norfolk",financial advisers norwich norfolk


We want our model to generate **industry-specific** slogans so a new **'modified_slogan'** column that adds the industry name to the front of processed slogan will be created.


In [ ]:
# Add industry to processed slogan
df['modified_slogan'] = df['industry'] + " " + df['processed_slogan']

# Check new column
df.head()

,industry,output,processed_slogan,modified_slogan
0,computer hardware,Taking Care of Small Business Technology,taking care of small business technology,computer hardware taking care of small busines...
1,"health, wellness and fitness",Build World-Class Recreation Programs,build world class recreation programs,"health, wellness and fitness build world class..."
2,internet,Most Powerful Lead Generation Software for Mar...,most powerful lead generation software for mar...,internet most powerful lead generation softwar...
3,internet,Hire quality freelancers for your job,hire quality freelancers for your job,internet hire quality freelancers for your job
4,financial services,"Financial Advisers Norwich, Norfolk",financial advisers norwich norfolk,financial services financial advisers norwich ...


## Build Training Input and Output Data for Generator


In [ ]:
# Build vocabulary and convert words to numerical indices
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["modified_slogan"])

# Vocabulary size
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index ordered by frequency
tokenizer.word_index

# Create increasingly longer input sequences for next-word prediction
input_sequences = []

for line in df["modified_slogan"]:
    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [ ]:
# Find the maximum sequence length
max_seq_len = len(max(input_sequences, key=lambda x: len(x)))
print(max_seq_len)

15


In [ ]:
# Pad input sequences
input_sequences = pad_sequences(input_sequences,
                                maxlen=max_seq_len, padding="pre")

In [ ]:
# Create X and y variables
X_gen = input_sequences[:, :-1]
y_gen = input_sequences[:, -1]

# Check shape of variables
print(f"Shape of X variable: {X_gen.shape}")
print(f"Shape of y variable: {y_gen.shape}")

Shape of X variable: (34633, 14)
Shape of y variable: (34633,)


In [ ]:
# One-hot encode y_gen
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

## Slogan Generator LSTM Model


In [ ]:
# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Model hyperparameters
input_dim = total_words
output_dim = 100
hidden_size_1 = 150
hidden_size_2 = 100
learning_rate = 0.001
num_epochs = 50

# Build slogan generator LSTM model
gen_model = Sequential()

# Add embedding layer
gen_model.add(Embedding(input_dim, output_dim,
                        input_length=(max_seq_len - 1)))

# Add LSTM layers
gen_model.add(LSTM(hidden_size_1, return_sequences=True))

gen_model.add(LSTM(hidden_size_2, return_sequences=False))

# Add dense output layer
gen_model.add(Dense(total_words, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile the model
gen_model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss="categorical_crossentropy",
                  metrics=['accuracy'])

In [ ]:
# Train the model
gen_model.fit(X_gen, y_gen, epochs=num_epochs, verbose=1)

Epoch 1/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.0701 - loss: 7.0562
Epoch 2/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.1024 - loss: 6.2656
Epoch 3/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.1378 - loss: 5.9020
Epoch 4/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.1728 - loss: 5.6187
Epoch 5/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - accuracy: 0.1916 - loss: 5.3800
Epoch 6/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.2054 - loss: 5.1465
Epoch 7/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.2176 - loss: 4.9191
Epoch 8/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.2297 - loss: 4.7027
Epoch 9/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.2420 - loss: 4.4897
Epoch 10/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.2537 - loss: 4.2762
Epoch 11/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - accuracy: 0.2704 - loss: 4.0717
Epoch 12/5

The model's training accuracy improves steadily from ~7% to ~79% over the 50 epochs. This shows the model was learning consistently during training.

## Slogan Generation using Model

In [ ]:
def generate_slogan(seed_text, max_words=20):
    """
    Generate slogan by predicting one word at a time from
    a seed text.

    Parameters:
        seed_text: starting text (industry name) to build the slogan
                   from.
        max_words: maximum number of words to generate.

    Returns:
        The generated slogan as a string.
    """
    for _ in range(max_words):

        # Tokenising and padding seed_text
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=(max_seq_len - 1),
                                   padding="pre")

        # Predict probability distribution of next word
        predictions = gen_model.predict(token_list, verbose=0)

        # Find index of most probable word
        predicted_index = np.argmax(predictions[0])

        output_word = None

        # Searching for the word that corresponds to the predicted index
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # If no valid word is found, algorithm stops
        if output_word is None:
            break

        # If a valid predicted word is found, append it to the seed text
        seed_text += " " + output_word

    return seed_text

In [ ]:
# Test function on different industries
print(f"\nComputer Hardware Slogan: "
      f"{generate_slogan('computer hardware')}")

print(f"\nInternet Slogan: "
      f"{generate_slogan('internet')}")

print(f"\nFinancial Services Slogan: "
      f"{generate_slogan('financial services')}")


Computer Hardware Slogan: computer hardware pc device analysis training design company in ranchi driven android cloud innovatives agency governments myway agency with queensland india usa

Internet Slogan: internet web design digital marketing agency in atlanta streaming seo digital agency blog account and enterprise management services volume malaysia from

Financial Services Slogan: financial services the lorawanâ standard in minnesota home trips opportunities in dubai uae freight dedicated provider for stock travel hvac equipment and


The generated slogans stay relatively on topic for their industry, using vocabulary genuinely associated with each one. However, the sentences themselves are not coherent, reading more like clusters of relevant words peppered within other words over a true structured slogan.

## Training Data for Slogan Classifier

In [ ]:
# Remove industries with one example
# Count how many examples per industry
industry_counts = df['industry'].value_counts()

# Find rows with more than one example
valid_row = df['industry'].map(industry_counts) > 1

# Filter df
df = df[valid_row]

Industries with only one example are removed from the dataset as the stratified train-test split used later requires at least two examples of each class to maintain proportional representation across both the training and test set. Since these industries only had a single slogan, they could not be split this way, so these six rows were removed before splitting.

## Build Training Input and Output Data for Classifier

In [ ]:
# Extract unique industries
industries = df['industry'].unique()

In [ ]:
# Empty dictionary
industry_to_index = {}

# Build a dictionary of unique industry index
for index, industry in enumerate(industries):
    industry_to_index[industry] = index

In [ ]:
# Add industry_index column to df
df['industry_index'] = df['industry'].map(industry_to_index)

In [ ]:
# Split df into training and test set
df_train, df_test = train_test_split(df, test_size=0.2,
                                     stratify=df["industry_index"])

# Check shape of training and test sets
print(f'Training Set Shape: {df_train.shape}')
print(f'Test Set Shape: {df_test.shape}')

Training Set Shape: (4256, 5)
Test Set Shape: (1064, 5)


In [ ]:
# Transform processed_slogan into sequences of numerical indices
X_train = tokenizer.texts_to_sequences(df_train['processed_slogan'])
X_test = tokenizer.texts_to_sequences(df_test['processed_slogan'])

In [ ]:
# Pad X_train and X_test sequences
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre")
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="pre")

In [ ]:
# One-hot encode industry index for training and test set
y_train = tf.keras.utils.to_categorical(df_train['industry_index'],
                                        num_classes=len(industries))
y_test = tf.keras.utils.to_categorical(df_test['industry_index'],
                                       num_classes=len(industries))

# Check shape of training and test target sets
print(f'Training Set Shape: {y_train.shape}')
print(f'Test Set Shape: {y_test.shape}')

Training Set Shape: (4256, 136)
Test Set Shape: (1064, 136)


## Slogan Classifier LSTM Model


In [ ]:
# Build LSTM model
class_model = Sequential()

# Add embedding layer
class_model.add(Embedding(input_dim, output_dim,
                          input_length=max_seq_len))

# Add LSTM layers
class_model.add(LSTM(hidden_size_1, return_sequences=True))

class_model.add(LSTM(hidden_size_2, return_sequences=False))

# Add dense output layer
class_model.add(Dense(len(industries), activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile the model
class_model.compile(optimizer=Adam(learning_rate=learning_rate),
                    loss="categorical_crossentropy",
                    metrics=['accuracy'])

In [ ]:
# Train the model
class_model.fit(X_train, y_train, epochs=num_epochs, verbose=1)

Epoch 1/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.0768 - loss: 4.3796
Epoch 2/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0811 - loss: 4.2896
Epoch 3/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0844 - loss: 4.2337
Epoch 4/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1273 - loss: 3.9051
Epoch 5/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2058 - loss: 3.4376
Epoch 6/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2951 - loss: 2.9788
Epoch 7/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3703 - loss: 2.5849
Epoch 8/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4366 - loss: 2.2702
Epoch 9/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4960 - loss: 2.0174
Epoch 10/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5623 - loss: 1.7874
Epoch 11/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6201 - loss: 1.5774
Epoch 12/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/s

The classifier's training accuracy rose quickly, reaching ~99% by epoch 30, and stayed roughly that level for the remaining epochs. This near-perfect training accuracy is a strong indicator of overfitting.

## Slogan Classification & Evaluation

In [ ]:
# Evaluate the model
class_model.evaluate(X_test, y_test)

34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2105 - loss: 7.1796


[7.179647445678711, 0.21052631735801697]

The class model performed poorly on the testing set, with an accuracy of 21%. This supports the idea that the model was overfit to the training data and has not learned to generalise well to unseen slogans.


This could perhaps be due to some industries having relatively few training examples, making it harder for the model to learn robust patterns for them. Additionally, several industries overlap in the broader field they belong to, which could be adding further confusion for the model.

To try and reduce this overfitting, L2 regularisation was applied to both LSTM layers. Penalties of 0.005 and 0.001 were attempted, however, neither addition resulted in an improvement to the test accuracy. Given these results, regularisation was not included in the final model, and the experimental code has been removed to keep the notebook focused on the final model.

In [ ]:
def classify_slogan(slogan):
    """
    Predict the industry based on a slogan

    Parameters:
        slogan: the raw slogan text to classify.

    Returns:
        The predicted industry.
    """
    # Clean input sequence
    slogan = preprocess_text(slogan)

    # Converting the slogan to a sequence of indices
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence
    padded_sequence = pad_sequences(sequence, maxlen=max_seq_len,
                                    padding='pre')

    # Get most probable industry from class_model
    prediction = class_model.predict(padded_sequence)

    # Assign index to most probable industry
    predicted_index = np.argmax(prediction[0])

    # Return the predicted industry
    return industries[predicted_index]

## Combining the two models

Generate a slogan for a company in the "internet" industry, then pass the generated slogan to the slogan classifier to see if it correctly classifies it as internet.

In [ ]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
Generated Slogan: internet web design digital marketing agency in atlanta streaming seo digital agency blog account and enterprise management services volume malaysia from
Predicted Industry: marketing and advertising


While the generated slogan loosely fits the internet industry, with words like 'web' and 'digital', it much more closely fits the marketing and advertising industry which the model classed it as. So although the classifier got the generated industry wrong, this suggests it was still picking up on genuine patterns in the text rather than guessing randomly.

This kind of overlap, where a slogan can plausibly belong to more than one industry, highlights how many industries commonly overlap in language, which likely affects how well the model can distinguish between them.